# Lab 2C: Peek inside an LLM: tokens, attention and sampling
**Presenter: Nagarjuna Naidu** | Time: ~20 min core

Learning outcomes
- See how text becomes tokens (and why tokens, not words, drive cost and limits).
- Compute self-attention by hand in numpy and read a heatmap.
- Control next-token sampling (temperature, top-k, top-p) and pick settings per task.

Everything runs offline; no big models.

In [ ]:
import sys, os, re, math, warnings, collections
import numpy as np, matplotlib
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
np.random.seed(7)
os.makedirs("artifacts", exist_ok=True)
INK, TEAL, LIME, CORAL, GOLD, PURPLE = "#0B1F2A", "#0E7C86", "#9BD13B", "#E4572E", "#F2B134", "#6C4AB6"
plt.rcParams.update({"axes.prop_cycle": plt.cycler(color=[TEAL, CORAL, GOLD, PURPLE, LIME, INK]),
                     "axes.edgecolor": INK, "axes.labelcolor": INK, "text.color": INK,
                     "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 100, "font.size": 10})
print("python", sys.version.split()[0], "| numpy", np.__version__, "| matplotlib", matplotlib.__version__)

## 1. Tokenisation
- Models read **tokens** (word pieces), not words.
- We borrow the WordPiece tokenizer of our embedding model. Production chat LLMs use other tokenizers (usually BPE), but the idea is identical.

In [ ]:
from fastembed import TextEmbedding
emb = TextEmbedding("BAAI/bge-small-en-v1.5")
tok = emb.model.tokenizer                      # a Hugging Face `tokenizers.Tokenizer`

para = ("Pharmacovigilance teams reconcile unbelievably messy adverse-event reports. "
        "Our retrieval-augmented chatbot answers questions about onboarding, VPN access and reimbursements.")
enc = tok.encode(para)
pieces = [t for t in enc.tokens if t not in ("[CLS]", "[SEP]")]
words = para.split()
print(f"{len(words)} words -> {len(pieces)} tokens  (ratio {len(pieces)/len(words):.2f} tokens/word)")
print(pieces)

In [ ]:
for w in ["onboarding", "reimbursements", "pharmacovigilance", "unbelievably", "VPN", "the"]:
    print(f"{w:20s} -> {tok.encode(w, add_special_tokens=False).tokens}")
print("\n'##' marks a piece that continues the previous one. Rare words split into several pieces; common words stay whole.")

### Token count -> cost
**Illustrative prices only** (not any vendor's real price list). Plug in your provider's numbers.

In [ ]:
PRICE_IN, PRICE_OUT = 0.50, 1.50          # ILLUSTRATIVE USD per 1M tokens
def cost(tokens_in, tokens_out, calls=1):
    return calls * (tokens_in * PRICE_IN + tokens_out * PRICE_OUT) / 1e6

# RAG call: question + 5 retrieved chunks of ~300 tokens + instructions, 250-token answer
t_in, t_out = 60 + 5 * 300 + 150, 250
print(f"one call: {t_in} in + {t_out} out = ${cost(t_in, t_out):.5f}")
for calls in (1_000, 50_000, 1_000_000):
    print(f"{calls:>9,} calls/month -> ${cost(t_in, t_out, calls):>10,.2f}   (illustrative)")
print("Levers: fewer/shorter chunks, shorter answers, cheaper model for easy steps, caching.")

### How subword vocabularies are built: a tiny BPE learner
Start from characters, repeatedly merge the most frequent adjacent pair.

In [ ]:
corpus = "low low low low low lower lower newest newest newest newest newest newest widest widest widest"
vocab = collections.Counter(" ".join(w) + " </w>" for w in corpus.split())   # {"l o w </w>": 5, ...}

def pair_counts(vocab):
    c = collections.Counter()
    for word, f in vocab.items():
        s = word.split()
        for a, b in zip(s, s[1:]): c[(a, b)] += f
    return c

def merge(vocab, pair):
    pat = re.compile(r"(?<!\S)" + re.escape(" ".join(pair)) + r"(?!\S)")
    return collections.Counter({pat.sub("".join(pair), w): f for w, f in vocab.items()})

merges = []
for step in range(8):
    pc = pair_counts(vocab)
    best = max(pc, key=pc.get)
    merges.append(best); vocab = merge(vocab, best)
    print(f"merge {step+1}: {best[0]!r} + {best[1]!r}  (count {pc[best]})")
print("final segmentation:", dict(vocab))

## 2. Token id -> embedding row (+ position)
Toy example: a **random** matrix, NOT the real model's weights. Real models learn this matrix in training.

In [ ]:
VOCAB, D = tok.get_vocab_size(), 8
E = np.random.randn(VOCAB, D) * 0.5            # embedding matrix: one row per token id (random here)
ids = tok.encode("the cat saw the dog", add_special_tokens=False)
print("tokens:", ids.tokens, "ids:", ids.ids)
X = E[ids.ids]                                  # lookup = row indexing
print("E shape", E.shape, "-> X shape", X.shape, "| row for 'the' equals E[%d]:" % ids.ids[0], np.allclose(X[0], E[ids.ids[0]]))

def pos_enc(n, d):                              # sinusoidal positions (original Transformer paper)
    p, i = np.arange(n)[:, None], np.arange(d // 2)[None, :]
    ang = p / (10000 ** (2 * i / d))
    return np.concatenate([np.sin(ang), np.cos(ang)], axis=1)
Xp = X + pos_enc(len(X), D)
print("'the' at position 0 vs 3 identical before positions?", np.allclose(X[0], X[3]), "| after?", np.allclose(Xp[0], Xp[3]))

## 3. Self-attention in numpy
- Each token makes a **query** (what am I looking for), **key** (what do I offer), **value** (what I pass on).
- weights = softmax(Q K^T / sqrt(d)); output = weights V.
- **Honest note:** these vectors are hand-designed (4 features) so the picture is readable. A real model learns them from data.

In [ ]:
sent = "The animal did not cross the street because it was too tired".split()
n = len(sent)
# hand-designed features per word: [animate, thing, pronoun/refers, other]
feat = {"animal": [1, 0, 0, 0], "street": [0, 1, 0, 0], "it": [0, 0, 1, 0], "tired": [.3, 0, 0, 0]}
X = np.array([feat.get(w.lower(), [0, 0, 0, 1]) for w in sent], float) + 0.05 * np.random.randn(n, 4)
Wq = np.array([[0, 0, 0, 0], [0, 0, 0, 0], [8.0, 0, 0, 0], [0, 1.0, 0, 0]])   # pronouns ask "who is animate?"; others look at 'other'
Wk = np.array([[1.0, 0, 0, 0], [0.3, 0, 0, 0], [0, 0, 0, 0], [0, 1.0, 0, 0]])  # animate nouns advertise "animate"
Wv = np.eye(4)
Q, K, V = X @ Wq, X @ Wk, X @ Wv
print("X", X.shape, "Q", Q.shape, "K", K.shape, "V", V.shape)

def softmax(z, axis=-1):
    z = z - z.max(axis=axis, keepdims=True); e = np.exp(z); return e / e.sum(axis=axis, keepdims=True)
scores = Q @ K.T / np.sqrt(K.shape[1])
W = softmax(scores)
out = W @ V
i_it = sent.index("it")
print("rows sum to 1:", np.allclose(W.sum(1), 1), "| output shape", out.shape)
print("'it' attends to:", sorted(zip(sent, [round(float(v), 2) for v in W[i_it]]), key=lambda t: -t[1])[:3])

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5.5))
im = ax.imshow(W, cmap="YlGnBu")
ax.set_xticks(range(n)); ax.set_xticklabels(sent, rotation=60, ha="right")
ax.set_yticks(range(n)); ax.set_yticklabels(sent)
ax.set_xlabel("attended-to word (key)"); ax.set_ylabel("current word (query)")
ax.set_title("Toy self-attention: 'it' looks at 'animal'\n(hand-designed vectors, not learned)")
plt.colorbar(im, label="attention weight"); plt.tight_layout()
plt.savefig("artifacts/day2_lab2c_attention.png"); plt.show()

**Multi-head attention**
- Several attention computations run in parallel, each with its own Wq/Wk/Wv, so different heads can track different relations (reference, syntax, position...).
- Head outputs are concatenated and mixed by one more matrix.

**Cost:** every token attends to every token, so compute and memory grow with n^2.

In [ ]:
ns = np.array([128, 512, 1024, 4096, 8192, 32768])
fig, ax = plt.subplots(1, 2, figsize=(9, 3.4))
ax[0].plot(ns, ns, color=TEAL, marker="o", label="n (linear)")
ax[0].plot(ns, ns**2 / 1e3, color=CORAL, marker="o", label="n^2 / 1000")
ax[0].set_xlabel("sequence length n"); ax[0].set_ylabel("relative work"); ax[0].legend(); ax[0].set_title("Linear vs quadratic")
ax[1].loglog(ns, ns**2, color=CORAL, marker="o"); ax[1].set_xlabel("n (tokens)"); ax[1].set_ylabel("attention pairs (n^2)")
ax[1].set_title("Log-log: 4x longer = 16x pairs")
plt.tight_layout(); plt.savefig("artifacts/day2_lab2c_quadratic.png"); plt.show()
print({int(k): f"{int(k)**2:,}" for k in ns})

## 4. Next-token sampling
The model outputs a **logit** per vocabulary token. Softmax with temperature turns them into probabilities; top-k / top-p trim the tail. Hand-made logits for "The meeting is scheduled for ..."

In [ ]:
toks = ["Monday", "Tuesday", "tomorrow", "noon", "Friday", "next", "9am", "midnight", "Sunday", "purple"]
logits = np.array([3.2, 2.9, 2.6, 2.0, 1.8, 1.2, 1.0, 0.2, 0.0, -2.0])

def probs(logits, T=1.0, top_k=None, top_p=None):
    p = softmax(logits / T)
    if top_k:
        p = np.where(p >= np.sort(p)[-top_k], p, 0)
    if top_p:
        o = np.argsort(-p); cum = np.cumsum(p[o])
        keep = o[: np.searchsorted(cum, top_p) + 1]
        m = np.zeros_like(p); m[keep] = 1; p = p * m
    return p / p.sum()

rng = np.random.default_rng(0)
fig, axs = plt.subplots(1, 3, figsize=(12, 3.4), sharey=True)
for a, T in zip(axs, (0.2, 1.0, 1.5)):
    p = probs(logits, T); a.bar(toks, p, color=TEAL); a.set_title(f"T = {T}"); a.tick_params(axis="x", rotation=70)
axs[0].set_ylabel("probability"); plt.tight_layout(); plt.savefig("artifacts/day2_lab2c_temperature.png"); plt.show()

In [ ]:
N = 1000
print(f"greedy (argmax): always '{toks[int(np.argmax(logits))]}'\n")
print(f"{'setting':22s}" + "".join(f"{t[:7]:>9s}" for t in toks) + "  distinct")
for name, kw in [("T=0.2", dict(T=0.2)), ("T=1.0", dict(T=1.0)), ("T=1.5", dict(T=1.5)),
                 ("T=1.0 top_k=3", dict(T=1.0, top_k=3)), ("T=1.5 top_p=0.9", dict(T=1.5, top_p=0.9))]:
    p = probs(logits, **kw); draws = rng.choice(len(toks), size=N, p=p)
    freq = np.bincount(draws, minlength=len(toks)) / N
    print(f"{name:22s}" + "".join(f"{f:9.3f}" for f in freq) + f"  {int((freq>0).sum()):>5d}")
print("\nLow T ~ greedy; high T flattens (rare tokens like 'purple' appear); top-k / top-p cut that tail.")

### Your turn 1: pick sampling settings
Choose temperature / top_p for each task and justify in one line.

In [ ]:
settings = {   # TODO: fill in (temperature, top_p, reason)
    "extraction (invoice fields to JSON)": (None, None, "TODO"),
    "email drafting":                      (None, None, "TODO"),
    "brainstorming product names":         (None, None, "TODO"),
}
# Hint: do you want the same answer every time, or variety?
for task, (t, p, why) in settings.items(): print(f"{task:38s} T={t} top_p={p}  {why}")

## 5. Recap
- LLMs see **tokens**; token counts set cost, latency and context limits.
- A token id just indexes an embedding row; position information is added on top.
- Attention = softmax(QK^T/sqrt(d))V: each token mixes information from the others; cost grows with n^2.
- Output is a probability distribution; temperature / top-k / top-p decide how adventurous the choice is (low for extraction, higher for creative work).